# Notebook 02 — EDA : Analyse Exploratoire des Données
## Projet Machine Learning : Healthy Illusion
**Détection des produits alimentaires à "image saine trompeuse"**

---

### Objectifs de ce notebook
1. Exploration univariée de chaque variable (histogrammes, boxplots, stats)
2. Exploration bivariée (variables vs cible `bad_nutrition`)
3. Analyse spécifique du déséquilibre des classes
4. Identification des features signaux (avec test statistique Mann-Whitney)
5. Analyse de la variable métier `healthy_illusion`
6. Synthèse et recommandations pour le preprocessing (Phase 2)


## 0. Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

# Style global
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Couleurs du projet
COLOR_0 = '#4CAF50'   # classe majoritaire — bon (nutriscore A/B/C)
COLOR_1 = '#F44336'   # classe minoritaire — mauvais (nutriscore D/E)
COLORS  = [COLOR_0, COLOR_1]

print(' Imports OK')


## 1. Chargement du dataset

Le dataset utilisé est le fichier final généré en Phase 1 : `data/dataset.csv`.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..')
DATASET_PATH = PROJECT_ROOT / 'data' / 'dataset.csv'

df = pd.read_csv(DATASET_PATH)
print(f' Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


## 2. Définition des variables

On distingue clairement :
- **Variable cible** : `bad_nutrition` (0 = bon nutriscore A/B/C, 1 = mauvais nutriscore D/E)
- **Variables numériques** : valeurs nutritionnelles pour 100g
- **Variables catégorielles** : catégorie, pays, labels
- **Colonnes exclues** : identifiants, variables dérivées (data leakage)


In [ ]:
# Variable cible
TARGET = 'bad_nutrition'

# Features numériques
NUM_COLS = [
    'sugars_100g', 'fat_100g', 'saturated_fat_100g',
    'salt_100g', 'fiber_100g', 'proteins_100g',
    'energy_kcal_100g', 'additives_count'
]

# Features catégorielles
CAT_COLS = ['main_category', 'country', 'has_labels', 'image_saine']

# Colonnes exclues de la modélisation (identifiants + leakage)
EXCLUDED = ['code', 'product_name', 'brands', 'nutriscore_grade', 'healthy_illusion']
# Justification des exclusions :
# - code / product_name / brands : identifiants, non informatifs
# - nutriscore_grade : construit directement depuis les nutriments → data leakage
# - healthy_illusion : variable dérivée de bad_nutrition + image_saine → data leakage

print(f'Cible             : {TARGET}')
print(f'Features num.     : {NUM_COLS}')
print(f'Features cat.     : {CAT_COLS}')
print(f'Colonnes exclues  : {EXCLUDED}')


## 3. Analyse des doublons

On vérifie les doublons exacts et les doublons sur le code produit (identifiant unique).


In [ ]:
# Doublons exacts (toutes colonnes identiques)
nb_dup_exact = df.duplicated().sum()
print(f'Doublons exacts         : {nb_dup_exact}')

# Doublons sur le code produit
nb_dup_code = df['code'].duplicated().sum()
print(f'Codes produits dupliqués: {nb_dup_code}')

if nb_dup_exact == 0 and nb_dup_code == 0:
    print(' Aucun doublon détecté.')
else:
    print(' Des doublons sont présents — à traiter en Phase 2 (03_preprocessing.ipynb).')


## 4. Valeurs manquantes

On quantifie le taux de valeurs manquantes par colonne et on propose une stratégie de traitement.
Les décisions finales seront appliquées dans `03_preprocessing.ipynb`.


In [ ]:
manq     = df.isnull().sum()
manq_pct = (manq / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Valeurs manquantes': manq,
    'Pourcentage (%)': manq_pct,
    'Stratégie proposée': [
        ' Aucune action' if p == 0
        else ('Supprimer ligne' if p < 5
        else ('Supprimer colonne' if p > 50
        else 'Imputer médiane/mode'))
        for p in manq_pct
    ]
}).sort_values('Pourcentage (%)', ascending=False)

missing_avec = missing_df[missing_df['Valeurs manquantes'] > 0]
if len(missing_avec) > 0:
    print(f'=== {len(missing_avec)} colonne(s) avec valeurs manquantes ===')
    display(missing_avec)

    # Graphique
    fig, ax = plt.subplots(figsize=(10, 4))
    manq_pct[manq_pct > 0].sort_values().plot(kind='barh', ax=ax, color='#E67E22')
    ax.axvline(5,  color='green', linestyle='--', linewidth=1.5, label='5% (seuil suppression ligne)')
    ax.axvline(50, color='red',   linestyle='--', linewidth=1.5, label='50% (seuil suppression colonne)')
    ax.set_title('Taux de valeurs manquantes par colonne', fontweight='bold')
    ax.set_xlabel('Pourcentage (%)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(' Aucune valeur manquante dans le dataset.')


## 5. Distribution de la cible — Analyse du déséquilibre

La variable cible est `bad_nutrition` :
- `0` : produit avec bon nutriscore (A, B ou C)
- `1` : produit avec mauvais nutriscore (D ou E)

Cette analyse est **obligatoire** selon le descriptif de la Phase 2.


In [ ]:
vc  = df[TARGET].value_counts().sort_index()
pct = df[TARGET].value_counts(normalize=True).sort_index() * 100

print('='*55)
print('ANALYSE DU DÉSÉQUILIBRE — bad_nutrition')
print('='*55)
print(f'\n  Total produits          : {len(df):,}')
print(f'  bad_nutrition = 0 (bon) : {vc[0]:,}  ({pct[0]:.1f}%)')
print(f'  bad_nutrition = 1 (mauvais) : {vc[1]:,}  ({pct[1]:.1f}%)')
print(f'\n  Ratio déséquilibre : 1 mauvais pour {vc[0]//vc[1]} bons')

ratio_min = pct.min()
if 5 <= ratio_min <= 25:
    print(f'\n  Classe minoritaire = {ratio_min:.1f}% — CONFORME (entre 5% et 25%)')
else:
    print(f'\n  Classe minoritaire = {ratio_min:.1f}% — NON CONFORME')


In [ ]:
# Visualisation : barres + camembert
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Barres ---
bars = axes[0].bar(
    ['0 — Bon (A/B/C)', '1 — Mauvais (D/E)'],
    [vc[0], vc[1]], color=COLORS, edgecolor='white', linewidth=1.5
)
axes[0].set_title('Distribution de bad_nutrition (effectifs)', fontweight='bold')
axes[0].set_ylabel('Nombre de produits')
for bar, val, p in zip(bars, [vc[0], vc[1]], [pct[0], pct[1]]):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + max(vc)*0.01,
        f'{val:,}\n({p:.1f}%)', ha='center', fontweight='bold', fontsize=11
    )
axes[0].set_ylim(0, max(vc)*1.18)

# --- Camembert ---
wedges, texts, autotexts = axes[1].pie(
    [vc[0], vc[1]],
    labels=['0 — Bon (A/B/C)', '1 — Mauvais (D/E)'],
    colors=COLORS, autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}, pctdistance=0.75
)
for at in autotexts:
    at.set_fontweight('bold')
    at.set_fontsize(12)
axes[1].set_title('Répartition des classes (en %)', fontweight='bold')

plt.suptitle('Déséquilibre des classes — bad_nutrition', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 6. Exploration univariée — Variables numériques

Pour chaque variable numérique : histogramme + boxplot côte à côte, avec moyenne et médiane annotées.


In [ ]:
# Tableau de statistiques descriptives complet
stats_df = pd.DataFrame({
    'Moyenne'    : df[NUM_COLS].mean().round(2),
    'Médiane'    : df[NUM_COLS].median().round(2),
    'Écart-type' : df[NUM_COLS].std().round(2),
    'Min'        : df[NUM_COLS].min().round(2),
    'Q1 (25%)'   : df[NUM_COLS].quantile(0.25).round(2),
    'Q3 (75%)'   : df[NUM_COLS].quantile(0.75).round(2),
    'Max'        : df[NUM_COLS].max().round(2),
})
print('=== Statistiques descriptives — Variables numériques ===')
display(stats_df)


In [ ]:
# Histogramme + boxplot côte à côte pour chaque variable numérique
fig, axes = plt.subplots(len(NUM_COLS), 2, figsize=(14, len(NUM_COLS)*3.2))

for i, col in enumerate(NUM_COLS):
    data = df[col].dropna()
    mean_val   = data.mean()
    median_val = data.median()

    # --- Histogramme ---
    axes[i, 0].hist(data, bins=50, color='#378ADD', edgecolor='white', alpha=0.85)
    axes[i, 0].axvline(mean_val,   color='red',    linestyle='--', linewidth=1.5,
                       label=f'Moyenne = {mean_val:.2f}')
    axes[i, 0].axvline(median_val, color='orange', linestyle='--', linewidth=1.5,
                       label=f'Médiane = {median_val:.2f}')
    axes[i, 0].set_title(f'{col} — Distribution', fontweight='bold')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Fréquence')
    axes[i, 0].legend(fontsize=9)

    # --- Boxplot ---
    bp = axes[i, 1].boxplot(data, vert=True, patch_artist=True,
                             boxprops=dict(facecolor='#E6F1FB'),
                             medianprops=dict(color='red', linewidth=2),
                             flierprops=dict(marker='o', markersize=3, alpha=0.3))
    axes[i, 1].set_title(f'{col} — Boxplot', fontweight='bold')
    axes[i, 1].set_ylabel(col)

plt.suptitle('Exploration univariée — Variables numériques', fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()


## 7. Exploration univariée — Variables catégorielles

Pour chaque variable catégorielle : fréquences, identification des **modalités rares** (< 1%).


In [ ]:
rare_summary = []

for col in CAT_COLS:
    vc_col  = df[col].value_counts()
    pct_col = df[col].value_counts(normalize=True) * 100
    rares   = pct_col[pct_col < 1]
    n_mod   = df[col].nunique()

    print(f"\n{'='*55}")
    print(f'Variable : {col}')
    print(f'Modalités uniques : {n_mod}')
    if len(rares) > 0:
        print(f' Modalités rares (< 1%) : {len(rares)} modalités → à regrouper en "other"')
    else:
        print(' Aucune modalité rare')

    # Top 15
    top15 = vc_col.head(15)
    fig, ax = plt.subplots(figsize=(12, 4))
    bars = ax.bar(range(len(top15)), top15.values, color='#378ADD', edgecolor='white')
    ax.set_xticks(range(len(top15)))
    ax.set_xticklabels(top15.index, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'Top 15 modalités — {col}', fontweight='bold')
    ax.set_ylabel('Nombre de produits')
    for bar, val in zip(bars, top15.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(top15)*0.01,
                str(val), ha='center', fontsize=8)
    plt.tight_layout()
    plt.show()

    rare_summary.append({
        'Variable': col,
        'Nb modalités totales': n_mod,
        'Nb modalités rares (<1%)': len(rares),
        '% modalités rares': round(len(rares)/n_mod*100, 1) if n_mod > 0 else 0
    })

print('\n=== Récapitulatif — Modalités rares ===')
display(pd.DataFrame(rare_summary))


## 8. Exploration bivariée — Variables numériques × cible

Boxplot de chaque variable par classe, avec **test statistique Mann-Whitney** pour valider la significativité.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    data0 = df[df[TARGET]==0][col].dropna()
    data1 = df[df[TARGET]==1][col].dropna()

    bp = axes[i].boxplot(
        [data0, data1],
        labels=['0 — Bon', '1 — Mauvais'],
        patch_artist=True,
        medianprops=dict(linewidth=2)
    )
    bp['boxes'][0].set_facecolor(COLOR_0 + '88')
    bp['boxes'][1].set_facecolor(COLOR_1 + '88')
    bp['medians'][0].set_color(COLOR_0)
    bp['medians'][1].set_color(COLOR_1)

    # Test Mann-Whitney
    stat, pval = stats.mannwhitneyu(data0, data1, alternative='two-sided')
    sig = '*** signal fort' if pval < 0.001 else ('** signal' if pval < 0.01 else 'ns')
    axes[i].set_title(f'{col}\np-value={pval:.2e} ({sig})', fontweight='bold', fontsize=9)

plt.suptitle('Variables numériques × bad_nutrition (boxplot par classe)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Statistiques moyennes par classe
df0 = df[df[TARGET]==0][NUM_COLS]
df1 = df[df[TARGET]==1][NUM_COLS]

stats_compare = pd.DataFrame({
    'Moy. bad=0 (bon)'     : df0.mean().round(2),
    'Moy. bad=1 (mauvais)' : df1.mean().round(2),
    'Diff. absolue'         : (df1.mean() - df0.mean()).round(2),
    'Ratio (mauvais/bon)'   : (df1.mean() / (df0.mean() + 0.001)).round(2),
})

print('=== Statistiques par classe ===')
print('Interprétation : Ratio > 1.5 ou < 0.7 = potentiel feature signal')
display(stats_compare)


## 9. Exploration bivariée — Variables catégorielles × cible

Tableau croisé et taux de classe positive (`bad_nutrition = 1`) par modalité.


In [ ]:
for col in ['main_category', 'image_saine', 'has_labels']:
    print(f"\n{'='*55}")
    print(f'Variable : {col} × {TARGET}')

    crosstab = pd.crosstab(df[col], df[TARGET], normalize='index') * 100
    crosstab.columns = ['% bad=0 (bon)', '% bad=1 (mauvais)']

    top = crosstab.sort_values('% bad=1 (mauvais)', ascending=False)
    if df[col].nunique() > 10:
        top = top.head(10)
    display(top.round(1))

    # Graphique
    fig, ax = plt.subplots(figsize=(12, 4))
    x = range(len(top))
    ax.bar(x, top['% bad=0 (bon)'],    color=COLOR_0, label='bad=0 (bon)',     alpha=0.8)
    ax.bar(x, top['% bad=1 (mauvais)'], color=COLOR_1, label='bad=1 (mauvais)', alpha=0.8,
           bottom=top['% bad=0 (bon)'])
    ax.set_xticks(x)
    ax.set_xticklabels(top.index, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'{col} × bad_nutrition (% par modalité)', fontweight='bold')
    ax.set_ylabel('%')
    ax.legend()
    plt.tight_layout()
    plt.show()


## 10. Matrice de corrélation — Variables numériques

Heatmap annotée avec `seaborn`. On identifie les variables fortement corrélées entre elles (|r| > 0.7).


In [ ]:
corr_matrix = df[NUM_COLS + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    ax=ax, linewidths=0.5, annot_kws={'size': 9}
)
ax.set_title('Matrice de corrélation — features numériques + cible', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Corrélations fortes entre features
print('\nCorrélations fortes (|r| > 0.7) entre features :')
found = False
for i, c1 in enumerate(NUM_COLS):
    for j, c2 in enumerate(NUM_COLS):
        if j <= i: continue
        r = corr_matrix.loc[c1, c2]
        if abs(r) > 0.7:
            print(f'  {c1} × {c2} → r = {r:.2f}   forte corrélation')
            found = True
if not found:
    print('  Aucune corrélation > 0.7 détectée entre les features.')


## 11. Identification des features signaux

Les **features signaux** sont les variables pour lesquelles la classe minoritaire (`bad_nutrition = 1`)
se comporte **clairement différemment** de la classe majoritaire.

Critères retenus :
- Test Mann-Whitney : **p-value < 0.001** (différence statistiquement significative)
- Ratio des moyennes : **> 1.3** ou **< 0.7** (différence pratique suffisante)


In [ ]:
print('=== FEATURES SIGNAUX — Test Mann-Whitney ===\n')

signals = []
for col in NUM_COLS:
    d0 = df[df[TARGET]==0][col].dropna()
    d1 = df[df[TARGET]==1][col].dropna()

    stat, pval = stats.mannwhitneyu(d0, d1, alternative='two-sided')
    ratio = d1.mean() / (d0.mean() + 1e-9)
    is_signal = pval < 0.001 and (ratio > 1.3 or ratio < 0.7)

    signals.append({
        'Feature'      : col,
        'Moy. bad=0'   : round(d0.mean(), 2),
        'Moy. bad=1'   : round(d1.mean(), 2),
        'Ratio'        : round(ratio, 2),
        'p-value'      : round(pval, 6),
        'Signal'       : ' OUI' if is_signal else '—'
    })

sig_df = pd.DataFrame(signals).sort_values('p-value')
display(sig_df)

sig_features = [r['Feature'] for r in signals if r['Signal'] == '✅ OUI']
print(f'\n{len(sig_features)} feature(s) signal(aux) identifiée(s) : {sig_features}')


In [ ]:
# Visualisation des features signaux : histogrammes superposés par classe
if sig_features:
    n = len(sig_features)
    ncols = min(n, 4)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if nrows > 1 else list(axes)

    for i, col in enumerate(sig_features):
        d0 = df[df[TARGET]==0][col].dropna()
        d1 = df[df[TARGET]==1][col].dropna()
        axes[i].hist(d0, bins=40, alpha=0.6, color=COLOR_0, label='bad=0 (bon)',     density=True)
        axes[i].hist(d1, bins=40, alpha=0.6, color=COLOR_1, label='bad=1 (mauvais)', density=True)
        axes[i].set_title(f'{col}\n Feature signal', fontweight='bold')
        axes[i].legend(fontsize=9)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Densité')

    # Masquer les axes vides
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Features signaux — Distribution par classe', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


## 12. Analyse métier — `image_saine` × `bad_nutrition`

La variable `image_saine` indique si un produit a **une apparence nutritionnelle positive** (catégorie
perçue saine, labels présents). En croisant avec `bad_nutrition`, on identifie les **"healthy illusions"** :
produits qui semblent sains mais ont en réalité un mauvais Nutri-Score.


In [ ]:
print('=== ANALYSE HEALTHY ILLUSION ===')
ct = pd.crosstab(df['image_saine'], df[TARGET])
ct.index = ['image_saine=0 (image pas saine)', 'image_saine=1 (image saine)']
ct.columns = ['bad=0 (bon nutriscore)', 'bad=1 (mauvais nutriscore)']
display(ct)

nb_image    = (df['image_saine'] == 1).sum()
nb_illusion = ((df['image_saine'] == 1) & (df[TARGET] == 1)).sum()
pct_ill     = nb_illusion / nb_image * 100 if nb_image > 0 else 0

print(f'\nParmi les {nb_image:,} produits avec image saine :')
print(f'  → {nb_illusion:,} ({pct_ill:.1f}%) ont un mauvais Nutri-Score = HEALTHY ILLUSION !')

# Graphique
ct_pct = pd.crosstab(df['image_saine'], df[TARGET], normalize='index') * 100
ct_pct.index = ["Pas d'image saine", 'Image saine']
ct_pct.columns = ['Bon (A/B/C)', 'Mauvais (D/E)']

fig, ax = plt.subplots(figsize=(8, 5))
ct_pct.plot(kind='bar', ax=ax, color=[COLOR_0, COLOR_1], edgecolor='white', width=0.6)
ax.set_title('image_saine × bad_nutrition (% par groupe)', fontweight='bold')
ax.set_ylabel('%')
ax.set_xticklabels(ct_pct.index, rotation=0)
ax.legend(['Bon nutriscore (A/B/C)', 'Mauvais nutriscore (D/E)'])
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x()+p.get_width()/2, p.get_height()+0.5),
                ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


## 13. Synthèse de l'EDA

Cette section résume les conclusions clés de l'analyse exploratoire et les actions à mener en Phase 2.


In [ ]:
vc_t  = df[TARGET].value_counts()
pct_t = df[TARGET].value_counts(normalize=True) * 100

print('='*65)
print('SYNTHÈSE EDA — PROJET HEALTHY ILLUSION')
print('='*65)

print(f'''
1. DATASET
   - {len(df):,} produits alimentaires collectés via Open Food Facts API
   - {len(NUM_COLS)} features numériques + {len(CAT_COLS)} features catégorielles
   - Cible : bad_nutrition (Nutri-Score D/E → 1)

2. DÉSÉQUILIBRE
   - bad_nutrition = 0 (bon)     : {vc_t[0]:,} produits ({pct_t[0]:.1f}%)
   - bad_nutrition = 1 (mauvais) : {vc_t[1]:,} produits ({pct_t[1]:.1f}%)
   - Ratio déséquilibre : {vc_t[0]//vc_t[1]}:1 → rééquilibrage nécessaire en Phase 3
   - Stratégies prévues : class_weight, SMOTE, RandomUnderSampler

3. FEATURES SIGNAUX IDENTIFIÉES
   - Variables les plus discriminantes (Mann-Whitney p<0.001) : voir section 11
   - Ces variables seront prioritaires lors du feature engineering

4. VARIABLES À TRAITER EN PREPROCESSING
   - Valeurs manquantes : product_name (~1.8%), brands (~2.7%) → imputation
   - Haute cardinalité : country (101 modalités), brands (4801) → regroupement
   - Encodage : nutriscore_grade (ordinal A→E), main_category (one-hot)
   - Data leakage à éviter : exclure nutriscore_grade et healthy_illusion comme features

5. FEATURE ENGINEERING ENVISAGÉ (≥ 2 features pour la Phase 2)
   - sugar_fat_ratio : ratio sucres/graisses (signal d'un profil "junk food masqué")
   - nutri_additives_score : combinaison nutriscore ordinal × nombre d'additifs
   - energy_density_class : binning de energy_kcal_100g en tranches
   - protein_fiber_score : score des "bons" nutriments (protéines + fibres)
''')
print('='*65)
print(' Dataset validé — prêt pour 03_preprocessing.ipynb')
print('='*65)
